# Process MAPseq FASTQs with pymutscan

**Audience:** researchers moving a MAPseq preprocessing workflow from RStudio/mutscan to Python.

**Prerequisites:** paired FASTQ structure, viral barcodes, RT sample indices, and UMIs.

**Learning goals:** generate a count database, correct sample indices and barcodes independently, collapse UMIs within barcode/sample strata, and inspect the resulting tables. The bundled reads are deterministic and synthetic.

## Outline

1. Locate or generate the synthetic FASTQs.
2. Stream reads into `raw_counts`.
3. Correct whitelisted RT indices.
4. Collapse 30-base barcodes by abundance and Hamming radius one.
5. Collapse UMIs within each corrected barcode/sample stratum.
6. Inspect and export processed tables.

In [ ]:
from __future__ import annotations

import json
import sqlite3
import subprocess
import sys
from pathlib import Path

from pymutscan import collapse_database, collapse_umis, digest_fastqs, map_sample_indices
from pymutscan.pipeline import export_table

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'examples' / 'synthetic'
OUTPUT = ROOT / 'notebooks' / 'output'
OUTPUT.mkdir(parents=True, exist_ok=True)
if not (DATA / 'example_R1.fastq.gz').exists():
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'generate_synthetic_mapseq.py')], check=True)
truth = json.loads((DATA / 'truth.json').read_text())
truth['total_read_pairs'], truth['layout']

## Build the processed database

The default layout is R1 = barcode(30) + constant(7), R2 = UMI(16) + RT index(6). Raw observations remain in SQLite; correction stages add mapping and derived tables.

In [ ]:
database = OUTPUT / 'example.sqlite'
for path in (database, Path(str(database) + '-wal'), Path(str(database) + '-shm')):
    if path.exists():
        path.unlink()

results = {}
results['digest'] = digest_fastqs(DATA / 'example_R1.fastq.gz', DATA / 'example_R2.fastq.gz', database)
results['indices'] = map_sample_indices(database, ['CGTGAT', 'ACATCG'], max_distance=1)
results['barcodes'] = collapse_database(database, collapse_max_dist=1, collapse_min_score=2)
results['umis'] = collapse_umis(database, collapse_max_dist=1)
results

## Inspect the data lineage

The row counts change because multiple observations acquire the same representative. Total supporting reads must remain conserved when `min_combo_reads=1`.

In [ ]:
con = sqlite3.connect(database)
tables = ['raw_counts', 'barcode_mapping', 'sample_index_mapping', 'collapsed_counts', 'umi_mapping', 'umi_collapsed_counts', 'molecule_counts']
table_rows = {table: con.execute(f'SELECT count(*) FROM {table}').fetchone()[0] for table in tables}
read_totals = {table: con.execute(f'SELECT sum(read_count) FROM {table}').fetchone()[0] for table in ['raw_counts', 'collapsed_counts', 'umi_collapsed_counts', 'molecule_counts']}
examples = con.execute('SELECT barcode, sample_index, molecule_count, read_count FROM molecule_counts ORDER BY read_count DESC LIMIT 6').fetchall()
con.close()
print('Rows:', table_rows)
print('Read totals:', read_totals)
print('Top barcode/sample molecule summaries:', examples)
assert len(set(read_totals.values())) == 1
assert read_totals['raw_counts'] == truth['expected_filters']['retained_reads']

In [ ]:
exported = {}
for table in ['qc', 'barcode_mapping', 'sample_index_mapping', 'molecule_counts']:
    target = OUTPUT / f'{table}.tsv.gz'
    exported[table] = export_table(database, table, target)
exported

## Exercise and pitfalls

**Exercise:** rerun barcode collapse with `min_combo_reads=2`. Predict which low-support observations disappear and verify the new read total.

**Common pitfall:** do not pass the Illumina UDI from the FASTQ header as the RT-index whitelist. They identify different pooling stages.

**Extension:** change `umi_length` and `sample_index_length` through `MapSeqConfig` for another fixed contiguous layout.